# UFC Fight Outcome Predictor

Binary classification model that predicts whether the **Red corner fighter wins** a UFC bout, using only pre-fight historical statistics.

**Validation approach:** Chronological ordering + TimeSeriesSplit CV to prevent temporal leakage. All preprocessing (imputation, scaling) is fit exclusively on training data via sklearn Pipeline.

---


## 1. Imports

In [ ]:
 import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              roc_curve, ConfusionMatrixDisplay)
from sklearn.calibration import CalibrationDisplay
from xgboost import XGBClassifier


## 2. Load Data

Four CSV files from the dataset:
- `UFC.csv` — aggregated fight + fighter stats per bout
- `fight_details.csv` — per-fight statistics and fighter names
- `event_details.csv` — event metadata
- `fighter_details.csv` — individual fighter profiles


In [ ]:
ufc      = pd.read_csv("data/UFC.csv")
fights   = pd.read_csv("data/fight_details.csv")
events   = pd.read_csv("data/event_details.csv")
fighters = pd.read_csv("data/fighter_details.csv")


## 3. Merge & Build Target

Fighter names (`r_name`, `b_name`) live in `fight_details.csv` and are needed to construct the binary target variable. We merge them into the main dataframe, then sort chronologically — a critical step to ensure all downstream splits respect temporal order.


In [ ]:
df = ufc.merge(
    fights[["fight_id", "r_name", "b_name"]],
    on="fight_id",
    how="left",
    suffixes=("", "_fights")
)
df = df.copy()

# Resolve column name after merge (handles potential suffix)
r_col = "r_name" if "r_name" in df.columns else "r_name_fights"

# Binary target: 1 = Red corner wins, 0 = Blue corner wins
df["red_win"] = (df["winner"] == df[r_col]).astype(int)

assert df["red_win"].sum() > 0, "red_win is all zeros — check column names"

# Sort chronologically to prevent temporal leakage in all subsequent splits
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Total fights : {len(df)}")
print(f"Red wins     : {df['red_win'].mean():.1%}")
print(f"Blue wins    : {1 - df['red_win'].mean():.1%}")


## 4. Feature Selection

Only pre-fight historical aggregates are used. An explicit whitelist is preferred over keyword-based filtering because it is robust to dataset changes — any new column must be added consciously rather than being silently included.

Features excluded: anything recording what happened *inside* the fight (strikes landed, takedowns in that bout, control time, knockdowns), which would constitute data leakage.


In [ ]:
PRE_FIGHT_FEATURES = [
    # Red corner — career historical aggregates
    "r_str_acc", "r_sapm", "r_splm", "r_str_def",
    "r_td_avg",  "r_td_avg_acc", "r_td_def", "r_sub_avg",
    # Blue corner — career historical aggregates
    "b_str_acc", "b_sapm", "b_splm", "b_str_def",
    "b_td_avg",  "b_td_avg_acc", "b_td_def", "b_sub_avg",
]

# Retain only columns present in this dataset version
PRE_FIGHT_FEATURES = [c for c in PRE_FIGHT_FEATURES if c in df.columns]

X = df[PRE_FIGHT_FEATURES].copy()
y = df["red_win"]

print(f"Features selected: {len(PRE_FIGHT_FEATURES)}")
print(PRE_FIGHT_FEATURES)


## 5. Feature Engineering — Differential Features

For each `(r_stat, b_stat)` pair, a differential feature `stat_diff = r_stat - b_stat` is constructed. Linear models benefit significantly from this: instead of discovering the relevant subtraction implicitly through two separate coefficients, the model receives the matchup advantage directly.

These features are arithmetic combinations of existing columns and do not introduce leakage regardless of split position.


In [ ]:
diff_cols = []
for col in PRE_FIGHT_FEATURES:
    if col.startswith("r_"):
        blue_col = col.replace("r_", "b_", 1)
        if blue_col in X.columns:
            diff_name = col.replace("r_", "", 1) + "_diff"
            X[diff_name] = X[col] - X[blue_col]
            diff_cols.append(diff_name)

print(f"Differential features added : {len(diff_cols)}")
print(f"Total features              : {X.shape[1]}")


## 6. Model Definition

Three classifiers are compared. A factory function `get_models()` is used instead of a shared dictionary so that each call returns fresh, unfitted instances — this prevents state from one training context (e.g. a CV fold) from leaking into another (e.g. the final split).


In [ ]:
def get_models():
    """Return a dict of fresh, unfitted model instances."""
    return {
        "Logistic Regression": LogisticRegression(max_iter=10000),
        "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42),
        "XGBoost":             XGBClassifier(n_estimators=200, random_state=42,
                                             eval_metric="logloss", verbosity=0),
    }


## 7. Model Selection — TimeSeriesSplit Cross-Validation

Five-fold temporal cross-validation is used for model selection. Each fold trains exclusively on past data and evaluates on the subsequent period, simulating real deployment conditions.

The standard deviation across folds is reported alongside the mean AUC to assess model stability.


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_results = {}

print(f"\n{'─'*65}")
print(f"  {'Model':<25} {'Mean AUC':>10} {'Std AUC':>10}  Per-fold AUC")
print(f"{'─'*65}")

for name, clf in get_models().items():
    fold_aucs = []

    for train_idx, test_idx in tscv.split(X):
        X_cv_train, X_cv_test = X.iloc[train_idx], X.iloc[test_idx]
        y_cv_train, y_cv_test = y.iloc[train_idx], y.iloc[test_idx]

        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler",  StandardScaler()),
            ("model",   clf)
        ])
        pipe.fit(X_cv_train, y_cv_train)
        probs_cv = pipe.predict_proba(X_cv_test)[:, 1]
        fold_aucs.append(roc_auc_score(y_cv_test, probs_cv))

    cv_results[name] = {
        "mean_auc":  np.mean(fold_aucs),
        "std_auc":   np.std(fold_aucs),
        "fold_aucs": fold_aucs,
    }

    fold_str = "  ".join([f"{v:.3f}" for v in fold_aucs])
    print(f"  {name:<25} {np.mean(fold_aucs):>10.3f} "
          f"{np.std(fold_aucs):>10.3f}  [{fold_str}]")

print(f"{'─'*65}")

best_name = max(cv_results, key=lambda k: cv_results[k]["mean_auc"])
std = cv_results[best_name]["std_auc"]
stability = ("stable ✓" if std < 0.02 else
             "moderate variance" if std < 0.04 else
             "high variance — interpret with caution")

print(f"\n  Best model : {best_name} (Mean AUC {cv_results[best_name]['mean_auc']:.3f})")
print(f"  Std AUC    : {std:.3f}  → {stability}")


## 8. Final Evaluation — Temporal Split 80/20

The best model from CV is retrained on the first 80% of fights (chronologically) and evaluated on the remaining 20%. This split is used for all visualizations and reported metrics.

A fresh model instance is used here — independent of any object used during CV.


In [ ]:
split_index = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

final_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  StandardScaler()),
    ("model",   get_models()[best_name])
])
final_pipeline.fit(X_train, y_train)

probs = final_pipeline.predict_proba(X_test)[:, 1]
preds = (probs > 0.5).astype(int)

baseline = y_test.value_counts(normalize=True).max()
accuracy = accuracy_score(y_test, preds)
auc      = roc_auc_score(y_test, probs)

print(f"\n{'─'*42}")
print(f"  Baseline (majority class) : {baseline:.3f}")
print(f"  Accuracy                  : {accuracy:.3f}")
print(f"  AUC-ROC                   : {auc:.3f}")
print(f"{'─'*42}")
print(f"  Train : {len(X_train)} fights")
print(f"  Test  : {len(X_test)} fights")


## 9. Evaluation Plots

Three plots are generated:
- **Model comparison:** mean AUC per model from CV, with standard deviation error bars
- **ROC curve:** true positive rate vs false positive rate for the best model on the final split
- **Confusion matrix:** classification breakdown on the final split


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Model comparison — CV AUC with error bars
model_names = list(cv_results.keys())
mean_aucs   = [cv_results[n]["mean_auc"] for n in model_names]
std_aucs    = [cv_results[n]["std_auc"]  for n in model_names]
bar_colors  = ["#2196F3", "#FF9800", "#4CAF50"]

axes[0].bar(model_names, mean_aucs, yerr=std_aucs,
            color=bar_colors, capsize=5, error_kw={"linewidth": 1.5})
axes[0].set_ylim(0.4, 1.0)
axes[0].set_ylabel("Mean AUC (TimeSeriesCV)")
axes[0].set_title("Model Comparison — CV")
axes[0].tick_params(axis="x", rotation=15)
for i, (v, s) in enumerate(zip(mean_aucs, std_aucs)):
    axes[0].text(i, v + s + 0.015, f"{v:.3f}", ha="center", fontsize=9)

# ROC Curve
fpr_, tpr_, _ = roc_curve(y_test, probs)
axes[1].plot(fpr_, tpr_, label=f"{best_name} (AUC={auc:.2f})", linewidth=2)
axes[1].plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — Final Split")
axes[1].legend()

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, preds,
    display_labels=["Blue wins", "Red wins"],
    ax=axes[2], colorbar=False
)
axes[2].set_title(f"Confusion Matrix — {best_name}")

plt.tight_layout()
plt.savefig("results/evaluation_plots.png", dpi=150)
plt.show()


## 10. Calibration Curve

AUC measures ranking quality but not probability reliability. A well-calibrated model should assign probabilities that match empirical frequencies: if it outputs 0.70 for Red winning, Red should win approximately 70% of those bouts.

A perfectly calibrated model follows the diagonal. Deviation above the diagonal indicates underconfidence; deviation below indicates overconfidence.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

CalibrationDisplay.from_predictions(
    y_test, probs,
    n_bins=10,
    ax=ax,
    name=best_name
)

ax.set_title(f"Calibration Curve — {best_name}\n(diagonal = perfectly calibrated)")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives (actual Red wins)")

plt.tight_layout()
plt.savefig("results/calibration_curve.png", dpi=150)
plt.show()


## 11. Feature Importance

Coefficients are used for Logistic Regression; Gini-based importance for tree models. Positive values indicate features that increase the predicted probability of Red winning; negative values favor Blue.


In [ ]:
best_clf      = final_pipeline.named_steps["model"]
feature_names = list(X_train.columns)

if hasattr(best_clf, "coef_"):
    importances = best_clf.coef_[0]
    imp_label   = "Coefficient (Logistic Regression)"
else:
    importances = best_clf.feature_importances_
    imp_label   = "Feature Importance (Gini)"

imp_series = pd.Series(importances, index=feature_names).sort_values()
imp_plot   = pd.concat([imp_series.head(8), imp_series.tail(8)])

fig, ax = plt.subplots(figsize=(9, 5))
imp_plot.plot(
    kind="barh", ax=ax,
    color=["#F44336" if v < 0 else "#2196F3" for v in imp_plot]
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"Feature Importance — {best_name}\n({imp_label})")
ax.set_xlabel(imp_label)
plt.tight_layout()
plt.savefig("results/feature_importance.png", dpi=150)
plt.show()
